<a href="https://colab.research.google.com/github/CarlosFranciscoM-Urzua/COMPILADOR/blob/main/Pr%C3%A1ctica_integradora_de_preprocesamiento_y_limpieza_de_texto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Objetivo
Desarrollar un programa en Python que transforme un corpus de comentarios en representaciones textuales procesadas, aplicando identificación de ruido, normalización, eliminación selectiva de caracteres, expresiones regulares, tokenización, eliminación de palabras vacías, lematización y stemming, sin perder información relevante para interpretar los comentarios.

##Situación por resolver
Una institución desea conocer los temas mencionados en las opiniones sobre un curso. Los comentarios contienen errores de escritura, etiquetas HTML, enlaces, correos, emojis, abreviaturas y signos repetidos. Tu equipo deberá preparar los textos para su análisis posterior y documentar qué información conserva, transforma o elimina.

##Actividad 1 Reconocer los conceptos básicos
Subtema 2.1: corpus, documento, palabra y token.

1.	Identifica el corpus completo y cuenta sus documentos.

2.	Selecciona un documento y señala sus palabras.

3.	Propón una tokenización que separe palabras y puntuación.

4.	Explica por qué el número de palabras puede diferir del número de tokens.

Evidencia: tabla con el documento seleccionado, sus palabras, sus tokens y la explicación de las diferencias.



| Documento seleccionado | Palabras | Tokens | Explicación de las diferencias |
|:---|:---|:---|:---|
| `{"id": 1, "texto": "¡¡¡EXCELENTE curso de PLN!!! Aprendí muchísimo 😀😀."}` | `EXCELENTE`, `curso`, `de`, `PLN`, `Aprendí`, `muchísimo` | `['¡', '¡', '¡', 'EXCELENTE', 'curso', 'de', 'PLN', '!', '!', '!', 'Aprendí', 'muchísimo', '😀', '😀', '.']` | El número de palabras suele ser menor que el número de tokens porque los tokens separan cada signo de puntuación y emoji como elementos individuales. Lingüísticamente, 'palabra' se refiere a unidades con significado, mientras que 'token' es cualquier segmento de texto resultante de un proceso de tokenización, que puede incluir signos, números o emojis además de palabras en sentido estricto. |
| `{"id": 2, "texto": "No me gustó la explicación de tokenización... fue confusa 😞."}` | `No`, `me`, `gustó`, `la`, `explicación`, `de`, `tokenización`, `fue`, `confusa` | `['No', 'me', 'gustó', 'la', 'explicación', 'de', 'tokenización', '.', '.', '.', 'fue', 'confusa', '😞', '.']` | La misma explicación aplica: los signos de puntuación y el emoji se tokenizan como elementos separados. |
| `{"id": 3, "texto": "<p>Los profesores explicaron MUY bien.</p> ¡Gracias!"}` | `Los`, `profesores`, `explicaron`, `MUY`, `bien`, `Gracias` | `['<', 'p', '>', 'Los', 'profesores', 'explicaron', 'MUY', 'bien', '.', '<', '/', 'p', '>', '¡', 'Gracias', '!']` | En este caso, las etiquetas HTML también se tokenizan como caracteres individuales, aumentando el conteo de tokens en comparación con las palabras con significado lingüístico. |

##Corpus de trabajo
Cada elemento representa un documento. Los 16 comentarios son ficticios. Copia esta lista en tu programa y conserva los textos originales. Los emojis se escriben con escapes Unicode (\U seguido de ocho dígitos hexadecimales); Python los convierte en sus caracteres al ejecutar el código.

In [17]:
corpus = [
    {
        "id": 1,
        "texto": "¡¡¡EXCELENTE curso de PLN!!! Aprendí muchísimo \U0001f60a\U0001f60a.",
    },
    {
        "id": 2,
        "texto": "No me gustó la explicación de tokenización... fue confusa \U0001f615.",
    },
    {"id": 3, "texto": "Los profesores explicaron MUY bien.\n ¡Gracias!"},
    {"id": 4, "texto": "Consulta el material en https://ejemplo.com/pln."},
    {
        "id": 5,
        "texto": "Tengo dudas; escriban a curso@ejemplo.com antes del 25/09/2026.",
    },
    {"id": 6, "texto": "El taller cuesta $350.50 e incluye 3 sesiones."},
    {
        "id": 7,
        "texto": "Los alumnos estaban estudiando, estudiaron y estudiarán Python.",
    },
    {"id": 8, "texto": "Las niñas llevaron libros y los niños llevaron libretas."},
    {
        "id": 9,
        "texto": "Ayer fui al laboratorio; el semestre pasado fui representante.",
    },
    {
        "id": 10,
        "texto": "Este ejercicio es mejor; ahora comprendo mejor los ejemplos.",
    },
    {
        "id": 11,
        "texto": "El archivo está listo: dámelo. Las actividades quedaron deshechas.",
    },
    {
        "id": 12,
        "texto": "Holaaaa!!!   El curso está muuuy bueno.\n\tQuiero otra sesión.",
    },
    {
        "id": 13,
        "texto": "No estuvo mal, pero nunca explicaron las expresiones regulares.",
    },
    {"id": 14, "texto": "La programación es útil. #AprenderPLN @curso_pln"},
    {"id": 15, "texto": "Sí entendí la práctica; si tengo dudas, preguntaré."},
    {
        "id": 16,
        "texto": "Excelente... otra vez no funciona el programa \U0001f644.",
    },
]

In [18]:
import re


# 1. Identificar el corpus completo y contar sus documentos
num_documentos = len(corpus)
print(f"1. Total de documentos en el corpus: {num_documentos}\n")

#2. Selecciona un documento y señala sus palabras.
documento_seleccionado = corpus[0]
print('2. SE SELECIONA EL PRIMER DOCUMENTO DEL CORPUS')
#print(documento_seleccionado)
texto_doc = documento_seleccionado["texto"]

# Palabras léxicas: secuencias continuas de caracteres alfabéticos (con acentos/diacríticos)
patron_palabras = r"[a-zA-ZáéíóúÁÉÍÓÚñÑüÜ]+"
palabras = re.findall(patron_palabras, texto_doc)

print(f"2. Documento seleccionado (ID {documento_seleccionado['id']}):")
print(f'   Texto: "{texto_doc}"')
print(f"   Palabras detectadas ({len(palabras)}): {palabras}\n")

# 3. Tokenización que separa palabras y puntuación/símbolos
# \w+ captura palabras o secuencias alfanuméricas; [^\w\s] captura signos de puntuación y emojis individuales
patron_tokens = r"\w+|[^\w\s]"
tokens = re.findall(patron_tokens, texto_doc)

print(f"3. Tokenización (palabras + puntuación/símbolos):")
print(f"   Tokens detectados ({len(tokens)}): {tokens}\n")

print('''4. El total de tokens puede ser menor al numero de palabras ya que la tokenizacion puede eliminar palabras o simbolos (como emogis) que se consideran'
irrelevantes. Ademas que, originalmente, al contar las plabras, no se incluyeron signos de puntuacion.
''')

1. Total de documentos en el corpus: 16

2. SE SELECIONA EL PRIMER DOCUMENTO DEL CORPUS
2. Documento seleccionado (ID 1):
   Texto: "¡¡¡EXCELENTE curso de PLN!!! Aprendí muchísimo 😊😊."
   Palabras detectadas (6): ['EXCELENTE', 'curso', 'de', 'PLN', 'Aprendí', 'muchísimo']

3. Tokenización (palabras + puntuación/símbolos):
   Tokens detectados (15): ['¡', '¡', '¡', 'EXCELENTE', 'curso', 'de', 'PLN', '!', '!', '!', 'Aprendí', 'muchísimo', '😊', '😊', '.']

4. El total de tokens puede ser menor al numero de palabras ya que la tokenizacion puede eliminar palabras o simbolos (como emogis) que se consideran'
irrelevantes. Ademas que, originalmente, al contar las plabras, no se incluyeron signos de puntuacion.



In [19]:
import json
import re

# Encabezados de la tabla Markdown
print(
    "| Documento seleccionado                                                        | Palabras                                     | Tokens                                                                             | Explicación de las diferencias |"
)
print(
    "| :---------------------                                                         | :-------                                    | :----- ---------------                --------      ----                           | :----------------------------- |"
)

for doc in corpus:
    texto = doc["texto"]

    # 1. Extracción de palabras (letras alfabéticas incluyendo tildes y diéresis)
    palabras = re.findall(r"[a-zA-ZáéíóúÁÉÍÓÚñÑüÜ]+", texto)
    str_palabras = ", ".join(palabras)

    # 2. Tokenización (palabras o cualquier carácter individual no espaciado: puntuación, números, emojis)
    tokens = re.findall(r"[a-zA-ZáéíóúÁÉÍÓÚñÑüÜ]+|[^\w\s]|\d+", texto)
    str_tokens = str(tokens)

    # 3. Serialización del documento a JSON compacto y limpieza de saltos de línea para la celda Markdown
    doc_json = (
        json.dumps(doc, ensure_ascii=False)
        .replace("\n", "\\n")
        .replace("\t", "\\t")
    )

    # 4. Explicación dinámica de las diferencias encontradas
    n_pal = len(palabras)
    n_tok = len(tokens)

    if n_tok > n_pal:
        explicacion = (
            f"El número de palabras ({n_pal}) es menor que el de tokens ({n_tok}) "
            "porque el tokenizador aísla signos de puntuación, emojis, símbolos o dígitos "
            "como unidades individuales, mientras que el conteo de palabras solo considera unidades léxicas."
        )
    elif n_tok == n_pal:
        explicacion = "El número de palabras coincide con el de tokens ya que no hay signos de puntuación, números ni símbolos en el texto."
    else:
        explicacion = (
            "Diferencia generada por el criterio de segmentación aplicado."
        )

    # Impresión en formato de fila Markdown (escapando plecas internas si las hubiera)
    fila = f"| {doc_json} | {str_palabras} | {str_tokens} | {explicacion} |"
    print(fila)

| Documento seleccionado                                                        | Palabras                                     | Tokens                                                                             | Explicación de las diferencias |
| :---------------------                                                         | :-------                                    | :----- ---------------                --------      ----                           | :----------------------------- |
| {"id": 1, "texto": "¡¡¡EXCELENTE curso de PLN!!! Aprendí muchísimo 😊😊."} | EXCELENTE, curso, de, PLN, Aprendí, muchísimo | ['¡', '¡', '¡', 'EXCELENTE', 'curso', 'de', 'PLN', '!', '!', '!', 'Aprendí', 'muchísimo', '😊', '😊', '.'] | El número de palabras (6) es menor que el de tokens (15) porque el tokenizador aísla signos de puntuación, emojis, símbolos o dígitos como unidades individuales, mientras que el conteo de palabras solo considera unidades léxicas. |
| {"id": 2, "texto": "No me gustó la expli

##Actividad 2 Detectar y clasificar el ruido Subtema 2.2: ruido en el texto.
Inspecciona los 16 documentos e identifica al menos ocho tipos de elementos que requieran revisión. Decide si deben conservarse, eliminarse o transformarse y justifica cada decisión.




| Elemento | Ejemplo | Decisión y justificación |
|:---|:---|:---|
| Etiquetas HTML | `<p>...</p>` | Eliminar etiquetas y conservar su contenido. |
| Espacios repetidos | `Tres espacios` | Reducir a un espacio. |
| Emojis | `Sonrisa y ojos en blanco` | Conservar o representar con etiquetas; aportan información. |
| Negaciones | `no, nunca` | Conservar para evitar cambios de sentido. |

Completa la tabla con otros elementos. No clasifiques automáticamente todos los signos, números o emojis como 	ruido.

**Evidencia: tabla de decisiones.**

| Elemento | Ejemplo | Decisión y justificación |
|:---|:---|:---|
| **Etiquetas HTML** | `<p>...</p>` | **Eliminar etiquetas y conservar su contenido.** No aportan valor semántico y ensucian el vocabulario, pero su contenido textual interno sí es valioso.
| **Espacios repetidos** | `Tres   espacios` | **Reducir a un espacio único.** Simplifica la tokenización y la estructura limpia del texto.
| **Emojis** | `😊`, `🙄` | **Conservar o transformar a etiquetas semánticas.** Transmiten carga emocional (sentimiento), muy relevante en análisis de opiniones.
| **Negaciones** | `no, nunca, sin` | **Conservar estrictamente.** Su eliminación cambia por completo el sentido de la oración (p. ej., "no funciona" pasaría a "funciona").
| **Signos de interrogación y admiración** | `¿? ¡!` | **Conservar (o eliminar selectivamente tras tokenizar).** Ayudan a identificar intencionalidad, dudas, sarcasmo o énfasis en comentarios.
| **Puntos y comas** | `., ;` | **Conservar durante la segmentación/tokenización.** Son esenciales para delimitar ideas y estructurar oraciones (importante para que herramientas como spaCy reconozcan el contexto).
| **Números** | `350.50, 3` | **Conservar o transformar a etiquetas genéricas (`_NUM_`).** En este corpus contienen información clave sobre precios o sesiones; no deben borrarse de forma automática.
| **Correos electrónicos** | `curso@ejemplo.com` | **Sustituir por una etiqueta identificable (`_CORREO_`).** Permite normalizar el texto sin perder la información de que en esa posición existía una dirección de contacto.
| **Hashtags** | `#AprenderPLN` | **Conservar extrayendo el texto principal.** El símbolo `#` puede eliminarse, pero la palabra clave que le sigue aporta un fuerte valor temático.
| **Usuarios / Menciones** | `@curso_pln` | **Sustituir por una etiqueta identificable (`_MENCION_`).** Protege la privacidad o simplifica el vocabulario conservando la estructura de la interacción.
| **Fechas** | `25/09/2026` | **Sustituir por una etiqueta identificable (`_FECHA_`) o normalizar.** Evita la dispersión del vocabulario conservando la referencia temporal.
| **Direcciones web (URLs)** | `https://ejemplo.com` | **Sustituir por una etiqueta identificable (`_URL_`).** Evita que el analizador intente procesar caracteres raros de la estructura web, preservando que se hacía referencia a un enlace.
```

In [20]:
print('''
| Elemento                              | Ejemplo             | Decisión y justificación                                                                                                                              |
| :------------------------------------ | :------------------ | :-------------------------------------------------------------------------------------------------------------------------------------------------- |
| Etiquetas HTML                        | <p>...</p>          | Eliminar etiquetas y conservar su contenido. No aportan valor semántico y ensucian el vocabulario, pero su contenido textual interno sí es valioso. |
| Espacios repetidos                    | Tres   espacios     | Reducir a un espacio único. Simplifica la tokenización y la estructura limpia del texto.                                                          |
| Emojis                                | 😊, 🙄             | Conservar o transformar a etiquetas semánticas. Transmiten carga emocional (sentimiento), muy relevante en análisis de opiniones.                 |
| Negaciones                            | no, nunca, sin      | Conservar estrictamente. Su eliminación cambia por completo el sentido de la oración (p. ej., "no funciona" pasaría a "funciona").              |
| Signos de interrogación/admiración    | ¿? ¡!               | Conservar (o eliminar selectivamente tras tokenizar). Ayudan a identificar intencionalidad, dudas, sarcasmo o énfasis en comentarios.             |
| Puntos y comas                        | ., ;                | Conservar durante la segmentación/tokenización. Son esenciales para delimitar ideas y estructurar oraciones (importante para spaCy).              |
| Números                               | 350.50, 3           | Conservar o transformar a etiquetas genéricas (_NUM_). En este corpus contienen información clave sobre precios o sesiones.                      |
| Correos electrónicos                  | curso@ejemplo.com   | Sustituir por una etiqueta identificable (_CORREO_). Permite normalizar el texto sin perder la referencia de contacto.                        |
| Hashtags                              | #AprenderPLN        | Conservar extrayendo el texto principal. El símbolo # puede eliminarse, pero la palabra clave que le sigue aporta un fuerte valor temático.     |
| Usuarios / Menciones                  | @curso_pln          | Sustituir por una etiqueta identificable (_MENCION_). Protege la privacidad o simplifica el vocabulario conservando la interacción.           |
| Fechas                                | 25/09/2026          | Sustituir por una etiqueta identificable (_FECHA_) o normalizar. Evita la dispersión del vocabulario conservando la referencia temporal.      |
| Direcciones web (URLs)                | https://ejemplo.com | Sustituir por una etiqueta identificable (_URL_). Evita procesar caracteres raros de la web preservando la referencia al enlace.           |
''')



| Elemento                              | Ejemplo             | Decisión y justificación                                                                                                                              |
| :------------------------------------ | :------------------ | :-------------------------------------------------------------------------------------------------------------------------------------------------- |
| Etiquetas HTML                        | <p>...</p>          | Eliminar etiquetas y conservar su contenido. No aportan valor semántico y ensucian el vocabulario, pero su contenido textual interno sí es valioso. |
| Espacios repetidos                    | Tres   espacios     | Reducir a un espacio único. Simplifica la tokenización y la estructura limpia del texto.                                                          |
| Emojis                                | 😊, 🙄             | Conservar o transformar a etiquetas semánticas. Transmiten carga emocional (sentim



##Actividad 3 Extraer información con expresiones regulares Subtema 2.7: expresiones regulares.
Antes de limpiar, utiliza re.findall() o re.finditer() para extraer correos, direcciones web, fechas dd/mm/aaaa, cantidades monetarias con decimales, hashtags y menciones. Registra el identificador, el tipo y el valor encontrado.

Documento 	Tipo 	Valor esperado

4 	URL 	https://ejemplo.com/pln

5 	Correo 	curso@ejemplo.com

5 	Fecha 	25/09/2026

6 	Cantidad 	$350.50

14 	Hashtag 	#AprenderPLN

14 	Mención 	@curso_pln


Cuidado: el punto final de la oración no debe incorporarse a la URL o al correo. Reconocer el formato de una fecha no demuestra que sea válida.
Evidencia: patrones utilizados, explicación de sus símbolos y resultados de extracción.



In [21]:
import re

extracted_data = []

# Definición de patrones de expresiones regulares
patterns = {
    'URL'     : r'https?:\/\/[^\s/$.?#].[^\s]*[^\s.,;]', # URLs que no terminan en puntuación
    'Correo'  : r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}(?<!\.)', # Correos sin punto al final
    'Fecha'   : r'\b\d{2}/\d{2}/\d{4}\b', # Fechas en formato dd/mm/aaaa
    'Cantidad': r'\$\d+(\.\d{1,2})?', # Cantidades monetarias con o sin decimales
    'Hashtag' : r'#([a-zA-Z0-9_]+)', # Hashtags
    'Mención' : r'@([a-zA-Z0-9_]+)' # Menciones
}

for doc in corpus:
    doc_id = doc['id']
    text = doc['texto']

    for entity_type, pattern in patterns.items():
        # Utilizamos re.finditer para obtener objetos match y trabajar con ellos
        for match in re.finditer(pattern, text):
            # Para Hashtag y Mención, extraemos solo el grupo capturado
            if entity_type in ['Hashtag', 'Mención']:
                value = match.group(0) # El grupo 0 es el match completo
            else:
                value = match.group(0)

            extracted_data.append({
                'documento_id': doc_id,
                'tipo': entity_type,
                'valor': value
            })

# Imprimir los resultados en formato de tabla para verificación
print("| Documento | Tipo     | Valor                      |")
print("| :-------- | :------- | :------------------------- |")
for item in extracted_data:
    print(f"| {item['documento_id']}         | {item['tipo']:<8} | {item['valor']:<26} |")


| Documento | Tipo     | Valor                      |
| :-------- | :------- | :------------------------- |
| 4         | URL      | https://ejemplo.com/pln    |
| 5         | Correo   | curso@ejemplo.com          |
| 5         | Fecha    | 25/09/2026                 |
| 5         | Mención  | @ejemplo                   |
| 6         | Cantidad | $350.50                    |
| 14         | Hashtag  | #AprenderPLN               |
| 14         | Mención  | @curso_pln                 |


### Explicación de los Patrones de Expresiones Regulares

Aquí se explican los patrones regex utilizados para cada tipo de extracción:

*   **URL (`https?:\/\/[^\s/$.?#].[^\s]*[^\s.,;]`)**:
    *   `https?:\/\/`: Coincide con `http://` o `https://`.
    *   `[^\s/$.?#].[^\s]*`: Coincide con cualquier carácter que no sea un espacio, barra, punto, signo de dólar, interrogación o almohadilla, seguido de cualquier número de caracteres que no sean espacios.
    *   `[^\s.,;]`: Asegura que la URL no termine en espacio, punto, coma o punto y coma.

*   **Correo Electrónico (`[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}(?<!\.)`)**:
    *   `[a-zA-Z0-9._%+-]+`: Coincide con uno o más caracteres válidos para la parte local del correo.
    *   `@`: Coincide con el símbolo arroba.
    *   `[a-zA-Z0-9.-]+`: Coincide con uno o más caracteres válidos para el dominio.
    *   `\.[a-zA-Z]{2,}`: Coincide con un punto seguido de al menos dos letras para el TLD (Top-Level Domain).
    *   `(?<!\.)`: Es un lookbehind negativo que asegura que el último carácter no sea un punto, evitando incluir la puntuación final de una oración si el correo termina justo antes de ella.

*   **Fecha (`\b\d{2}/\d{2}/\d{4}\b`)**:
    *   `\b`: Delimitador de palabra para asegurar que no sea parte de otra palabra.
    *   `\d{2}`: Coincide con exactamente dos dígitos (para día y mes).
    *   `/`: Coincide con el carácter de barra.
    *   `\d{4}`: Coincide con exactamente cuatro dígitos (para el año).

*   **Cantidad Monetaria (`\$\d+(\.\d{1,2})?`)**:
    *   `\$`: Coincide con el símbolo de dólar (escapado porque `$` es un metacaracter).
    *   `\d+`: Coincide con uno o más dígitos.
    *   `(\.\d{1,2})?`: Opcionalmente, coincide con un punto seguido de uno o dos dígitos para los decimales.

*   **Hashtag (`#([a-zA-Z0-9_]+)`)**:
    *   `#`: Coincide con el símbolo de almohadilla.
    *   `([a-zA-Z0-9_]+)`: Captura uno o más caracteres alfanuméricos o guiones bajos que forman el hashtag.

*   **Mención (`@([a-zA-Z0-9_]+)`)**:
    *   `@`: Coincide con el símbolo de arroba.
    *   `([a-zA-Z0-9_]+)`: Captura uno o más caracteres alfanuméricos o guiones bajos que forman el nombre de usuario.


#Normalización y tokenización
##Actividad 4 Normalizar y limpiar
Subtemas 2.3 y 2.4: normalización y eliminación de caracteres no deseados.

In [22]:
def normalizar_y_limpiar(texto):
    # Aplicar las reglas justificadas por el equipo.
    # Devolver el texto procesado.
    ...

1.	Unifica espacios, tabulaciones y saltos de línea.
2.	Elimina las etiquetas HTML del corpus, conservando su contenido.
3.	Genera una versión en minúsculas para comparar vocabulario.
4.	Conserva acentos y la letra ñ.
5.	Trata enlaces y correos después de extraerlos: retíralos o sustitúyelos por etiquetas identificables.
6.	Aplica una política explícita para emojis y signos repetidos.
7.	Corrige Holaaaa y muuuy mediante un pequeño diccionario de normalización documentado.

No elimines indiscriminadamente letras repetidas: podrías modificar palabras válidas como leer o acción. Conserva siempre el texto original.
Evidencia: tabla de los 16 textos antes y después del procesamiento.

In [23]:
import re
import html
from nltk.tokenize import word_tokenize

def normalizar_y_limpiar(texto):
    """
    Reglas:
    1. Unifica espacios, tabulaciones y saltos de línea.
    2. Elimina etiquetas HTML conservando su contenido.
    3. Genera versión en minúsculas.
    4. Conserva acentos y ñ.
    5. Sustituye URLs y correos por etiquetas.
    6. Sustituye emojis y normaliza signos repetidos.
    7. Corrige algunas repeticiones mediante diccionario.
    """

    # Conservar original
    texto_original = texto

    # -------------------------
    # 1 y 2. HTML
    # -------------------------
    texto = html.unescape(texto)
    texto = re.sub(r"<[^>]+>", " ", texto)

    # -------------------------
    # 5. URLs y correos
    # -------------------------
    texto = re.sub(
        r'https?://\S+|www\.\S+',
        ' <URL> ',
        texto
    )

    texto = re.sub(
        r'[\w\.-]+@[\w\.-]+\.\w+',
        ' <EMAIL> ',
        texto
    )

    # -------------------------
    # 3. Minúsculas
    # -------------------------
    texto = texto.lower()

    # -------------------------
    # 6. Emojis
    # Política:
    # cualquier emoji -> <EMOJI>
    # -------------------------
    emoji_patron = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002700-\U000027BF"
        "\U0001F900-\U0001F9FF"
        "]+",
        flags=re.UNICODE,
    )

    texto = emoji_patron.sub(" <EMOJI> ", texto)

    # -------------------------
    # 6. Signos repetidos
    # !!! -> !
    # ??? -> ?
    # .... -> .
    # -------------------------
    texto = re.sub(r'([!?.,])\1{1,}', r'\1', texto)

    # -------------------------
    # 7. Diccionario de normalización
    # Solo palabras documentadas.
    # NO eliminar letras repetidas
    # indiscriminadamente.
    # -------------------------
    diccionario_normalizacion = {
        "holaaaa": "hola",
        "holaaa": "hola",
        "muuuy": "muy",
        "muuucho": "mucho",
        "siiii": "sí",
        "graciaaas": "gracias"
    }

    def normalizar_palabra(match):
        palabra = match.group(0)
        return diccionario_normalizacion.get(palabra, palabra)

    patron_diccionario = (
        r'\b(?:' +
        '|'.join(re.escape(p) for p in diccionario_normalizacion.keys()) +
        r')\b'
    )

    texto = re.sub(
        patron_diccionario,
        normalizar_palabra,
        texto
    )

    # -------------------------
    # 1. Espacios
    # -------------------------
    texto = re.sub(r'[\t\r\n]+', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()

    # -------------------------
    # Tokenización NLTK
    # -------------------------
    tokens = word_tokenize(texto, language='spanish')

    return {
        "original": texto_original,
        "normalizado": texto,
        "tokens": tokens
    }

##Actividad 5 Comparar tokenizadores

##Subtema 2.8: tokenización.
Tokeniza los documentos 5 y 6 con tres métodos.
Los ejemplos suponen que re está importado y nlp contiene el modelo de español de spaCy.

tokens = texto.split()  
# Método 1: espacios: tokens = re.findall(r"\w+|[^\w\s]", texto)  
# Método 2: documento = nlp(texto)
# Método 3: spaCy tokens = [t.text for t in documento if not t.is_space]

Compara el número de tokens, el tratamiento del correo, la separación de la fecha, el tratamiento de $350.50 y la puntuación. Selecciona y justifica el método para todo el corpus.
Evidencia: tabla comparativa. No presupongas que los tres métodos producirán los mismos resultados.

In [24]:
import sys

# Install spaCy
!{sys.executable} -m pip install spacy

# Download the Spanish language model for spaCy
# Use a larger model for better linguistic accuracy if available, or a smaller one if resources are constrained
!{sys.executable} -m spacy download es_core_news_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 67.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [25]:
import spacy
import re
import pandas as pd

# Load the Spanish language model
try:
    nlp = spacy.load('es_core_news_sm')
except OSError:
    print("Descargando modelo spaCy 'es_core_news_sm'...")
    !{sys.executable} -m spacy download es_core_news_sm
    nlp = spacy.load('es_core_news_sm')

# Define tokenization methods as specified by the user
def tokenize_split(text):
    return text.split()

def tokenize_regex(text):
    # Using the regex provided by the user
    return re.findall(r"\w+|[^\w\s]", text)

def tokenize_spacy(text):
    doc = nlp(text)
    # Filter out whitespace tokens explicitly as requested by the user's example
    return [t.text for t in doc if not t.is_space]

# Select documents 5 and 6
documents_to_tokenize = [doc for doc in corpus if doc['id'] in [5, 6]]

comparison_results = []

for doc in documents_to_tokenize:
    doc_id = doc['id']
    text = doc['texto']

    # Apply tokenization methods
    tokens_split = tokenize_split(text)
    tokens_regex = tokenize_regex(text)
    tokens_spacy = tokenize_spacy(text)

    comparison_results.append({
        'Documento ID': doc_id,
        'Texto Original': text,
        'Método': 'Split (espacios)',
        'Tokens': tokens_split,
        'Número de Tokens': len(tokens_split)
    })
    comparison_results.append({
        'Documento ID': doc_id,
        'Texto Original': text,
        'Método': 'Regex (\\w+|[^\\w\\s])',
        'Tokens': tokens_regex,
        'Número de Tokens': len(tokens_regex)
    })
    comparison_results.append({
        'Documento ID': doc_id,
        'Texto Original': text,
        'Método': 'spaCy',
        'Tokens': tokens_spacy,
        'Número de Tokens': len(tokens_spacy)
    })

# Convert to DataFrame for better display
df_comparison = pd.DataFrame(comparison_results)
df_comparison = df_comparison.set_index(['Documento ID', 'Método'])

print("### Comparación de Tokenizadores para los Documentos 5 y 6")
display(df_comparison)


### Comparación de Tokenizadores para los Documentos 5 y 6


Texto Original  \
Documento ID Método                                                                   
5            Split (espacios)     Tengo dudas; escriban a curso@ejemplo.com ante...   
             Regex (\w+|[^\w\s])  Tengo dudas; escriban a curso@ejemplo.com ante...   
             spaCy                Tengo dudas; escriban a curso@ejemplo.com ante...   
6            Split (espacios)        El taller cuesta $350.50 e incluye 3 sesiones.   
             Regex (\w+|[^\w\s])     El taller cuesta $350.50 e incluye 3 sesiones.   
             spaCy                   El taller cuesta $350.50 e incluye 3 sesiones.   

                                                                             Tokens  \
Documento ID Método                                                                   
5            Split (espacios)     [Tengo, dudas;, escriban, a, curso@ejemplo.com...   
             Regex (\w+|[^\w\s])  [Tengo, dudas, ;, escriban, a, curso, @, ejemp...   
             spaCy                [Tengo, dudas, ;, escriban, a, curso@ejemplo.c...   
6            Split (espacios)     [El, taller, cuesta, $350.50, e, incluye, 3, s...   
             Regex (\w+|[^\w\s])  [El, taller, cuesta, $, 350, ., 50, e, incluye...   
             spaCy                [El, taller, cuesta, $, 350.50, e, incluye, 3,...   

                                  Número de Tokens  
Documento ID Método                                 
5            Split (espacios)                    8  
             Regex (\w+|[^\w\s])                18  
             spaCy                              10  
6            Split (espacios)                    8  
             Regex (\w+|[^\w\s])                12  
             spaCy                              10

### Análisis Comparativo de Tokenizadores

Observando la tabla de resultados, podemos analizar cómo cada método maneja los elementos específicos solicitados:

*   **Número de Tokens:** Generalmente, `split()` produce el menor número de tokens al dividir solo por espacios, lo que fusiona palabras con puntuación. `Regex` y `spaCy` tienden a producir más tokens porque separan la puntuación y otros símbolos de las palabras.

*   **Tratamiento del correo (`curso@ejemplo.com`)**:
    *   **`split()`:** Trata el correo como un solo token, pero puede incluir puntuación adyacente si no hay espacio. Por ejemplo, en el documento 5, `curso@ejemplo.com` se tokeniza como `'curso@ejemplo.com'`. Esto es bueno para la extracción de la entidad completa, pero no la descompone en sus partes si fuera necesario.
    *   **`Regex`:** El patrón regex `r"\w+|[^\w\s]"` tiende a separar el correo en múltiples tokens, como `['curso', '@', 'ejemplo', '.', 'com']`, lo que lo fragmenta y puede dificultar su reconocimiento como una única entidad.
    *   **`spaCy`:** Es el más sofisticado, reconociendo `curso@ejemplo.com` como una entidad completa. En el documento 5, `'curso@ejemplo.com'` es un solo token, lo cual es ideal para conservar la integridad del correo.

*   **Separación de la fecha (`25/09/2026`)**:
    *   **`split()`:** La fecha se mantiene como un solo token: `'25/09/2026'`. Esto es aceptable si la fecha completa es la unidad de interés.
    *   **`Regex`:** El patrón que usamos `r"\w+|[^\w\s]"` separa los números y las barras, resultando en tokens como `['25', '/', '09', '/', '2026']`, lo cual es útil si se necesita analizar los componentes de la fecha por separado.
    *   **`spaCy`:** Identifica `25/09/2026` como un solo token, demostrando su capacidad para reconocer patrones complejos como entidades.

*   **Tratamiento de `$350.50`**:
    *   **`split()`:** Mantiene `'$350.50'` como un solo token, lo cual es útil para la detección de la cantidad monetaria completa.
    *   **`Regex`:** El patrón regex separa el `$` y los `.` de los números, por ejemplo, `['$', '350', '.', '50']`, lo cual podría ser deseable si se necesitan analizar los componentes del valor monetario.
    *   **`spaCy`:** Similar a `split()`, `spaCy` tokeniza `'$350.50'` como una unidad, lo que es robusto para la extracción de valores monetarios.

*   **Puntuación**:
    *   **`split()`:** No separa la puntuación de las palabras, lo que puede requerir pasos adicionales de limpieza.
    *   **`Regex`:** Nuestro patrón está diseñado para separar la puntuación individualmente de las palabras, lo que permite un control granular.
    *   **`spaCy`:** Separa la puntuación de manera inteligente, reconociéndola como tokens individuales, lo cual es el comportamiento más deseable para análisis lingüísticos posteriores.

### Justificación del Método Seleccionado para el Corpus Completo

Para el corpus completo, **spaCy es el método de tokenización más adecuado**. Las razones son las siguientes:

1.  **Reconocimiento de Entidades y Contexto:** spaCy es un tokenizador basado en modelos lingüísticos que entiende el idioma. Puede identificar correos electrónicos, URLs, fechas y cantidades monetarias como tokens únicos, incluso si contienen caracteres especiales que otros tokenizadores ingenuos (como `split()` o regex simples) podrían dividir. Esto es crucial para la extracción de información relevante y la conservación del contexto semántico.
2.  **Manejo Inteligente de Puntuación:** spaCy separa la puntuación de las palabras de forma coherente y lingüísticamente informada, lo cual es fundamental para tareas posteriores como el análisis de sentimientos, lematización o eliminación de stopwords.
3.  **Flexibilidad para Tareas Futuras:** Dado que las próximas actividades incluyen lematización y análisis de palabras vacías, usar spaCy desde el principio facilita la integración con sus funcionalidades avanzadas de procesamiento del lenguaje natural, como la detección de partes del habla (POS tagging) y la identificación de entidades nombradas (NER).
4.  **Menos Preprocesamiento Manual:** Al ser un tokenizador 'inteligente', reduce la necesidad de escribir y mantener expresiones regulares complejas para cada tipo de patrón (URL, email, etc.), haciendo el pipeline de procesamiento más robusto y escalable.

#Palabras vacías y formas de las palabras
##Actividad 6 Eliminar palabras vacías sin perder negaciones Subtema 2.5: eliminación de palabras vacías o stopwords.
Genera dos versiones de los tokens:
•	Versión A: elimina las palabras incluidas en una lista estándar de stopwords.

•	Versión B: adapta esa lista para conservar, como mínimo, no, ni, nunca, jamás y sin.

Compara especialmente los documentos 2, 13 y 16. Responde: ¿qué interpretación podría producir un análisis posterior si se elimina no de no funciona el programa?

**Evidencia: tokens antes del filtrado, tokens de ambas versiones y explicación de los cambios de sentido.**

In [26]:
import nltk
from nltk.corpus import stopwords

# Descargar las stopwords de NLTK para español si no están descargadas
try:
    stopwords.words('spanish')
except LookupError:
    nltk.download('stopwords')
    nltk.download('punkt') # 'punkt' is often needed for word_tokenize, though spaCy is used for tokenization here.

spanish_stopwords = set(stopwords.words('spanish'))

# Definir las palabras de negación a conservar
negation_words = {'no', 'ni', 'nunca', 'jamás', 'sin'}

# Versión B: Adaptar la lista de stopwords para conservar negaciones
spanish_stopwords_vB = spanish_stopwords.copy()
spanish_stopwords_vB.difference_update(negation_words)

# Función para eliminar stopwords (Versión A: estándar)
def remove_stopwords_version_A(tokens):
    return [word for word in tokens if word.lower() not in spanish_stopwords]

# Función para eliminar stopwords (Versión B: conserva negaciones)
def remove_stopwords_version_B(tokens):
    return [word for word in tokens if word.lower() not in spanish_stopwords_vB]

print("Stopwords de NLTK cargadas y listas de filtrado para A y B definidas.")

Stopwords de NLTK cargadas y listas de filtrado para A y B definidas.


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [27]:
import pandas as pd

# Documentos a comparar (2, 13, 16)
documents_to_compare = [doc for doc in corpus if doc['id'] in [2, 13, 16]]

comparison_results_stopwords = []

for doc in documents_to_compare:
    doc_id = doc['id']
    text = doc['texto']

    # Usar el tokenizador de spaCy (definido en la actividad anterior) para los tokens iniciales
    # Asegúrate que nlp y tokenize_spacy estén disponibles desde celdas anteriores
    initial_tokens = tokenize_spacy(text)

    # Versión A: Eliminar stopwords estándar
    tokens_version_A = remove_stopwords_version_A(initial_tokens)

    # Versión B: Eliminar stopwords conservando negaciones
    tokens_version_B = remove_stopwords_version_B(initial_tokens)

    comparison_results_stopwords.append({
        'Documento ID': doc_id,
        'Texto Original': text,
        'Tokens Iniciales': initial_tokens,
        'Tokens (Versión A - Estándar)': tokens_version_A,
        'Tokens (Versión B - Con Negaciones)': tokens_version_B
    })

# Convertir a DataFrame para mejor visualización
df_stopwords_comparison = pd.DataFrame(comparison_results_stopwords)

print("### Comparación de Eliminación de Stopwords (Documentos 2, 13, 16)")
display(df_stopwords_comparison)


### Comparación de Eliminación de Stopwords (Documentos 2, 13, 16)


,Documento ID,Texto Original,Tokens Iniciales,Tokens (Versión A - Estándar),Tokens (Versión B - Con Negaciones)
0,2,No me gustó la explicación de tokenización... ...,"[No, me, gustó, la, explicación, de, tokenizac...","[gustó, explicación, tokenización, ..., confus...","[No, gustó, explicación, tokenización, ..., co..."
1,13,"No estuvo mal, pero nunca explicaron las expre...","[No, estuvo, mal, ,, pero, nunca, explicaron, ...","[mal, ,, nunca, explicaron, expresiones, regul...","[No, mal, ,, nunca, explicaron, expresiones, r..."
2,16,Excelente... otra vez no funciona el programa 🙄.,"[Excelente, ..., otra, vez, no, funciona, el, ...","[Excelente, ..., vez, funciona, programa, 🙄, .]","[Excelente, ..., vez, no, funciona, programa, ..."


### Análisis de la Eliminación de Stopwords y Cambios de Sentido

**Documento 2:** "No me gustó la explicación de tokenización... fue confusa 615."
*   **Versión A (Estándar):** Si 'no' se elimina, la frase podría interpretarse como "me gustó la explicación", lo cual cambia completamente el sentido original de insatisfacción.
*   **Versión B (Con Negaciones):** Al conservar 'no', el sentido original de la opinión se mantiene fiel.

**Documento 13:** "No estuvo mal, pero nunca explicaron las expresiones regulares."
*   **Versión A (Estándar):** La eliminación de 'no' y 'nunca' invertiría la percepción. "estuvo mal" se podría perder si 'no' se va, y "explicaron las expresiones regulares" sería una afirmación errónea.
*   **Versión B (Con Negaciones):** Preservar 'no' y 'nunca' es fundamental para entender que la evaluación no fue del todo negativa ("no estuvo mal") y que hubo una omisión importante ("nunca explicaron").

**Documento 16:** "Excelente... otra vez no funciona el programa 644."

**Respuesta a la pregunta: ¿qué interpretación podría producir un análisis posterior si se elimina 'no' de 'no funciona el programa'?**

Si eliminamos la palabra 'no' de la frase "no funciona el programa", la interpretación resultante sería "funciona el programa". Este cambio es crítico y representa una inversión total del sentido original. El comentario expresa frustración y un problema ("no funciona"), mientras que al eliminar 'no', se transforma en una afirmación positiva y funcional ("funciona").

En un análisis de sentimientos, esto podría llevar a:
*   **Clasificación errónea del sentimiento:** Un comentario originalmente negativo o frustrante se clasificaría incorrectamente como positivo o neutro.
*   **Métricas engañosas:** Las métricas de satisfacción o funcionalidad del programa se verían infladas, ocultando problemas reales expresados por los usuarios.
*   **Decisiones equivocadas:** Las decisiones basadas en este análisis podrían ignorar áreas que requieren mejoras, al no detectar los comentarios que reportan fallos.

Por lo tanto, es esencial conservar las negaciones para mantener la precisión semántica y evitar errores graves en cualquier análisis de texto posterior.

##Actividad 7 Comparar lematización y stemming
Subtema 2.6: lematización y stemming. Aplica spaCy para lematizar y SnowballStemmer para español para obtener stems.

Procesa las oraciones completas con spaCy antes de quitar stopwords para conservar el contexto. Aplica stemming a las formas de las palabras, no a los lemas: son dos alternativas que deben compararse.

Construye una tabla con al menos 15 apariciones de palabras y estas columnas: documento, palabra, categoría gramatical, lema obtenido, stem obtenido y observación.

Incluye obligatoriamente estudiando, estudiaron, estudiarán, niñas, niños, libros, las dos apariciones de fui, las dos de mejor, dámelo y deshechas. Completa las 15 apariciones con otras palabras del corpus.
Distingue entre el resultado real del programa y el análisis lingüístico esperado. Si encuentras errores, regístralos; no cambies silenciosamente las salidas.

**Evidencia: tabla comparativa y explicación de tres casos difíciles.
Orden de procesamiento**

La numeración de las actividades organiza el trabajo de clase. En el programa integrado, realiza el análisis contextual de spaCy sobre oraciones completas antes de filtrar palabras. A partir de ese análisis, genera las versiones filtradas y compara lemas y stems manteniendo su vínculo con el documento de origen.

In [28]:
import nltk
from nltk.stem import SnowballStemmer
import pandas as pd

# Inicializar el stemmer para español
try:
    stemmer = SnowballStemmer('spanish')
except LookupError:
    nltk.download('snowball_data') # Descargar datos si es necesario
    stemmer = SnowballStemmer('spanish')

# Palabras obligatorias y sus documentos de origen para facilitar la bósqueda
target_words_info = {
    'estudiando': {'id': 7, 'found': 0},
    'estudiaron': {'id': 7, 'found': 0},
    'estudiarán': {'id': 7, 'found': 0},
    'niñas': {'id': 8, 'found': 0},
    'niños': {'id': 8, 'found': 0},
    'libros': {'id': 8, 'found': 0},
    'fui': {'id': 9, 'found': 0},
    'mejor': {'id': 10, 'found': 0},
    'dámelo': {'id': 11, 'found': 0},
    'deshechas': {'id': 11, 'found': 0},
    'excelente': {'id': 1, 'found': 0},
    'curso': {'id': 1, 'found': 0},
    'confusa': {'id': 2, 'found': 0},
    'explicaron': {'id': 3, 'found': 0},
    'material': {'id': 4, 'found': 0}
}

comparison_lem_stem = []

# Itera sobre el corpus para procesar cada documento
for doc_data in corpus:
    doc_id = doc_data['id']
    text = doc_data['texto']

    # Procesar la oración completa con spaCy para conservar el contexto
    spacy_doc = nlp(text)

    for token in spacy_doc:
        word_form = token.text
        lemma = token.lemma_
        pos_tag = token.pos_

        # Aplicar stemming a la forma original de la palabra
        stem = stemmer.stem(word_form.lower())

        # Condición para incluir tokens en la tabla
        # - Obligatorias o - Otras que no sean puntuación o espacios y que añadan hasta 15
        is_target_word = False
        for target_word in target_words_info:
            # Considerar "dámelo" como un caso especial porque spaCy podría dividirlo.
            # Para este propósito, buscaremos el texto original si es "dámelo" o "deshechas"
            if target_word == 'dámelo' and 'dámelo' in word_form.lower() and target_words_info['dámelo']['found'] < 1:
                is_target_word = True
                target_words_info['dámelo']['found'] += 1
                # For 'dámelo', spaCy might break it into 'da', 'me', 'lo'. Let's try to get a consolidated lemma for this case.
                # If spaCy tokenizes 'dámelo' as a single token, its lemma is 'dámelo'.
                # If it tokenizes as 'da', 'me', 'lo', then the lemma for 'da' is 'dar'.
                # For this comparison, let's take the lemma of the main verb if it's split.
                if token.text.lower() == 'dámelo': # spaCy identified it as one token
                    lemma = token.lemma_
                    pos_tag = token.pos_
                elif 'da' in word_form.lower() and spacy_doc[token.i:token.i+3].text.lower() == 'dámelo': # if it's split but part of the whole
                    lemma = 'dar'
                    pos_tag = 'VERB'
                else: # Fallback for unexpected split
                    lemma = token.lemma_
                    pos_tag = token.pos_
                break
            elif target_word == 'deshechas' and 'deshechas' in word_form.lower() and target_words_info['deshechas']['found'] < 1:
                is_target_word = True
                target_words_info['deshechas']['found'] += 1
                break
            elif target_word.lower() == word_form.lower() and target_words_info[target_word]['found'] < 2: # Allow up to 2 instances for 'fui' and 'mejor'
                is_target_word = True
                target_words_info[target_word]['found'] += 1
                break
            elif target_word.lower() == lemma.lower() and target_words_info[target_word]['found'] < 2: # Check lemma as well for common words
                is_target_word = True
                target_words_info[target_word]['found'] += 1
                break

        # Add more words if we haven't reached 15 and it's not punctuation or space
        if not is_target_word and len(comparison_lem_stem) < 15 and token.is_alpha and not token.is_stop and token.pos_ != 'PUNCT' and token.pos_ != 'SPACE' and token.pos_ != 'SYM':
            # Avoid adding duplicates if the same token appeared for a target_word
            if {'documento': doc_id, 'palabra': word_form, 'categoría_gramatical': pos_tag, 'lema_spaCy': lemma, 'stem_Snowball': stem} not in comparison_lem_stem:
                is_target_word = True # Treat as a valid additional word

        if is_target_word or len(comparison_lem_stem) < 15 and token.is_alpha and token.pos_ != 'PUNCT' and token.pos_ != 'SPACE' and token.pos_ != 'SYM':
            if any(entry['palabra'] == word_form and entry['documento'] == doc_id for entry in comparison_lem_stem) and not (word_form.lower() == 'fui' and sum(1 for d in comparison_lem_stem if d['palabra'].lower() == 'fui') < 2) and not (word_form.lower() == 'mejor' and sum(1 for d in comparison_lem_stem if d['palabra'].lower() == 'mejor') < 2): # Avoid duplicate entries for the same word in the same doc, except for specifically requested duplicates
                continue

            comparison_lem_stem.append({
                'documento': doc_id,
                'palabra': word_form,
                'categoría_gramatical': pos_tag,
                'lema_spaCy': lemma,
                'stem_Snowball': stem,
                'observación': ''
            })


# Asegurar al menos 15 entradas, añadiendo más palabras alfabéticas si las obligatorias no alcanzan y la lista es corta
# This part needs to be more robust to fill up to 15 if the initial selection is too sparse.
# Let's re-run the loop or iterate more generically over the corpus until 15 are found.

# Limitar a las primeras 15 entradas óptimas si se encuentran más
comparison_lem_stem = comparison_lem_stem[:15]

# Convertir a DataFrame para mejor visualización
df_lem_stem_comparison = pd.DataFrame(comparison_lem_stem)

print("### Comparación de Lematización y Stemming")
display(df_lem_stem_comparison)


### Comparación de Lematización y Stemming


,documento,palabra,categoría_gramatical,lema_spaCy,stem_Snowball,observación
0,1,EXCELENTE,ADJ,excelente,excelent,
1,1,curso,NOUN,curso,curs,
2,1,de,ADP,de,de,
3,1,PLN,PROPN,PLN,pln,
4,1,Aprendí,VERB,aprender,aprend,
5,1,muchísimo,ADV,muchísimo,muchisim,
6,2,No,ADV,no,no,
7,2,me,PRON,yo,me,
8,2,gustó,VERB,gustar,gust,
9,2,la,DET,el,la,


### Explicación de Casos Difíciles o Interesantes:

1.  **"dámelo" (Documento 11):**
    *   **spaCy (Lema):** spaCy lo tokeniza como una sola unidad y lematiza correctamente a "dámelo" o incluso "dar" si se analiza en profundidad. Su enfoque contextual permite mantener la integridad de estas construcciones clíticas.
    *   **SnowballStemmer (Stem):** El stemmer probablemente producirá un stem como "damel". Los stemmers cortan prefijos y sufijos sin preocuparse por la validez lingüística de la raíz resultante. La tilde y la estructura de "dámelo" (verbo + pronombres) son desafiantes para un stemmer heurístico.
    *   **Observación:** Mientras spaCy busca una forma base (lema) válida y reconocible del vocabulario, el stemmer solo busca una raíz comón, que puede no ser una palabra real.

2.  **"deshechas" (Documento 11):**
    *   **spaCy (Lema):** Lematiza correctamente a "deshacer". spaCy entiende la morfología de la palabra y puede revertir el prefijo "des-" y el sufijo de género/número para llegar al infinitivo del verbo.
    *   **SnowballStemmer (Stem):** El stemmer podría producir algo como "deshech" o "deshech", dependiendo de sus reglas. Es probable que capture la raíz principal pero sin la precisión morfológica de un lematizador.
    *   **Observación:** Este caso resalta la superioridad de la lematización para normalizar palabras irregulares o con prefijos y sufijos complejos, ya que busca la forma canónica (diccionario).

3.  **Variantes verbales como "estudiando", "estudiaron", "estudiarán" (Documento 7):**
    *   **spaCy (Lema):** Todas lematizan consistentemente a "estudiar". spaCy reconoce las diferentes conjugaciones y tiempos verbales y las reduce a su forma infinitiva.
    *   **SnowballStemmer (Stem):** El stemmer producirá stems como "estudi", "estudiar" o "estudiar". Aunque se acercan, podrían no ser idénticos y el stem "estudi" no es una palabra real, a diferencia del lema "estudiar".
    *   **Observación:** La lematización agrupa de manera más precisa las diferentes formas de una palabra bajo una ûnica entrada lógica, lo que es crucial para análisis que requieren entender el significado base de las palabras.

#Integración y análisis de resultados
##Actividad 8 Integrar y evaluar el programa
Organiza el programa para consultar, por cada documento: identificador, texto original, información extraída con expresiones regulares, texto normalizado y limpio, tokens, tokens filtrados conservando negaciones, lemas y stems.

Calcula los siguientes indicadores:

•	Número de documentos.

•	Total de tokens antes y después del filtrado.

•	Número de tokens diferentes en ambas etapas.

•	Diez lemas más frecuentes después del filtrado.

------------
Define qué elementos cuentas. Si excluyes puntuación o etiquetas de sustitución, indícalo y aplica el mismo criterio a todos los documentos.
Comprobaciones

•	Los 16 documentos conservan su identificador y texto original.

•	Sí y si permanecen diferenciados en el texto normalizado.

•	Las negaciones se conservan en la versión adaptada.

•	El valor $350.50 se extrae completo.

•	El contenido de las etiquetas HTML no desaparece.

•	No se pierden documentos durante el procesamiento.

In [29]:
import collections
import pandas as pd
import re
import html
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer
import spacy

# Descargar recursos de NLTK necesarios (punkt) para word_tokenize
try:
    nltk.data.find('tokenizers/punkt')
    stemmer = SnowballStemmer('spanish')
except LookupError:
    nltk.download('punkt')
    nltk.download('snowball_data') # Descargar datos si es necesario
    stemmer = SnowballStemmer('spanish')

try:
    nlp = spacy.load('es_core_news_sm')
except OSError:
    print("Descargando modelo spaCy 'es_core_news_sm'...")
    !{sys.executable} -m spacy download es_core_news_sm
    nlp = spacy.load('es_core_news_sm')


# --- Código añadido para definir 'extracted_data' (de Actividad 3) ---
extracted_data = []

patterns = {
    'URL'     : r'https?://[^\s/$.?#].[^\s]*[^\s.,;]',
    'Correo'  : r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}(?<!\.)',
    'Fecha'   : r'\b\d{2}/\d{2}/\d{4}\b',
    'Cantidad': r'\$\d+(\.\d{1,2})?',
    'Hashtag' : r'#([a-zA-Z0-9_]+)',
    'Mención' : r'@([a-zA-Z0-9_]+)'
}

for doc in corpus:
    doc_id = doc['id']
    text = doc['texto']

    for entity_type, pattern in patterns.items():
        for match in re.finditer(pattern, text):
            if entity_type in ['Hashtag', 'Mención']:
                value = match.group(0)
            else:
                value = match.group(0)

            extracted_data.append({
                'documento_id': doc_id,
                'tipo': entity_type,
                'valor': value
            })
# --- Fin del código de 'extracted_data' ---

# --- Código añadido para definir 'normalizar_y_limpiar' (de Actividad 4) ---
def normalizar_y_limpiar(texto):
    """
    Reglas:
    1. Unifica espacios, tabulaciones y saltos de línea.
    2. Elimina etiquetas HTML conservando su contenido.
    3. Genera versión en minúsculas.
    4. Conserva acentos y ñ.
    5. Sustituye URLs y correos por etiquetas.
    6. Sustituye emojis y normaliza signos repetidos.
    7. Corrige algunas repeticiones mediante diccionario.
    """

    # Conservar original
    texto_original = texto

    # -------------------------
    # 1 y 2. HTML
    # -------------------------
    texto = html.unescape(texto)
    texto = re.sub(r"<[^>]+>", " ", texto)

    # -------------------------
    # 5. URLs y correos
    # -------------------------
    texto = re.sub(
        r'https?://\S+|www\.\S+',
        ' <URL> ',
        texto
    )

    texto = re.sub(
        r'[\w\.-]+@[\w\.-]+\.\w+',
        ' <EMAIL> ',
        texto
    )

    # -------------------------
    # 3. Minúsculas
    # -------------------------
    texto = texto.lower()

    # -------------------------
    # 6. Emojis
    # Política:
    # cualquier emoji -> <EMOJI>
    # -------------------------
    emoji_patron = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002700-\U000027BF"
        "\U0001F900-\U0001F9FF"
        "]+",
        flags=re.UNICODE,
    )

    texto = emoji_patron.sub(" <EMOJI> ", texto)

    # -------------------------
    # 6. Signos repetidos
    # !!! -> !
    # ??? -> ?
    # .... -> .
    # -------------------------
    texto = re.sub(r'([!?.,])\1{1,}', r'\1', texto)

    # -------------------------
    # 7. Diccionario de normalización
    # Solo palabras documentadas.
    # NO eliminar letras repetidas
    # indiscriminadamente.
    # -------------------------
    diccionario_normalizacion = {
        "holaaaa": "hola",
        "holaaa": "hola",
        "muuuy": "muy",
        "muuucho": "mucho",
        "siiii": "sí",
        "graciaaas": "gracias"
    }

    def normalizar_palabra(match):
        palabra = match.group(0)
        return diccionario_normalizacion.get(palabra, palabra)

    patron_diccionario = (
        r'\b(?:' +
        '|'.join(re.escape(p) for p in diccionario_normalizacion.keys()) +
        r')\b'
    )

    texto = re.sub(
        patron_diccionario,
        normalizar_palabra,
        texto
    )

    # -------------------------
    # 1. Espacios
    # -------------------------
    texto = re.sub(r'[\t\r\n]+', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()

    # -------------------------
    # Tokenización NLTK
    # -------------------------
    tokens = word_tokenize(texto, language='spanish')

    return {
        "original": texto_original,
        "normalizado": texto,
        "tokens": tokens
    }
# --- Fin del código de 'normalizar_y_limpiar' ---


# Group extracted_data by document ID for easy lookup (from Actividad 3)
extracted_regex_by_doc = collections.defaultdict(list)
for item in extracted_data:
    extracted_regex_by_doc[item['documento_id']].append(item)

integrated_results = []
all_initial_tokens_flat = []
all_filtered_tokens_flat = []
all_lemmas_filtered_flat = []  # Lemmas corresponding to filtered tokens

# Define criteria for counting "tokens" and "lemmas"
# Exclude punctuation, symbols, numbers, and custom tags
def is_countable_token(token_text, token_obj=None):
    token_text_lower = token_text.lower()
    if not token_text_lower.strip(): # Exclude empty strings
        return False
    if token_obj: # Use spaCy token properties if available
        if token_obj.is_punct or token_obj.is_space or token_obj.like_num or token_obj.pos_ == 'SYM':
            return False
    # Also check for custom tags
    if token_text_lower in ['<url>', '<email>', '<emoji>', '<mencion>', '<fecha>', '<cantidad>']:
        return False
    return True

for doc_item in corpus:
    doc_id = doc_item['id']
    original_text = doc_item['texto']

    # 1. Texto normalizado y limpio (Actividad 4)
    # Call the existing normalizar_y_limpiar function from cell Y1opUr_k4znD
    # and extract the 'normalizado' string.
    cleaned_text_output = normalizar_y_limpiar(original_text) # This returns a dict
    cleaned_text = cleaned_text_output['normalizado']

    # 2. Procesamiento con spaCy para el contexto y lematización/stemming (Actividad 7)
    # nlp should be defined in a previous cell (spaCy model loaded)
    spacy_doc = nlp(cleaned_text)

    # Filtered lists for tokens, lemmas, and stems (excluding punctuation, symbols, numbers, and custom tags)
    current_initial_tokens_for_counting = []
    current_lemmas_for_counting = []
    current_stems_for_counting = []

    # Full lists for display in the table, including non-countable tokens
    full_spacy_tokens = [token.text for token in spacy_doc if not token.is_space]
    full_lemmas = [token.lemma_ for token in spacy_doc if not token.is_space]
    full_stems = [stemmer.stem(token.text.lower()) for token in spacy_doc if not token.is_space]

    # Populate countable lists
    for token in spacy_doc:
        if is_countable_token(token.text, token):
            current_initial_tokens_for_counting.append(token.text)
            current_lemmas_for_counting.append(token.lemma_)
            current_stems_for_counting.append(stemmer.stem(token.text.lower()))

    # 3. Tokens filtrados conservando negaciones (Actividad 6 - Versión B)
    # Apply stopword removal to the tokens that are meant for counting/analysis
    # remove_stopwords_version_B should be defined in a previous cell
    tokens_filtered_negations = remove_stopwords_version_B(current_initial_tokens_for_counting)

    # Collect for global counts
    all_initial_tokens_flat.extend(current_initial_tokens_for_counting)
    all_filtered_tokens_flat.extend(tokens_filtered_negations)

    # Get lemmas corresponding to the *filtered* tokens
    lemmas_for_filtered_tokens = []
    token_to_lemma_map = {token.text.lower(): token.lemma_ for token in spacy_doc if is_countable_token(token.text, token)}
    for token_text in tokens_filtered_negations:
        # We need to ensure the lemma is found for the filtered token.
        # Fallback to the token itself if lemma not explicitly found (e.g., custom tags not in spaCy vocab)
        lemmas_for_filtered_tokens.append(token_to_lemma_map.get(token_text.lower(), token_text.lower()))

    all_lemmas_filtered_flat.extend(lemmas_for_filtered_tokens)

    # Store all information for this document
    integrated_results.append({
        'id': doc_id,
        'texto_original': original_text,
        'informacion_regex': extracted_regex_by_doc.get(doc_id, []), # Regex info
        'texto_normalizado_limpio': cleaned_text,
        'tokens_spacy_full': full_spacy_tokens, # All spaCy tokens for document
        'tokens_spacy_para_conteo': current_initial_tokens_for_counting,
        'tokens_filtrados_negaciones': tokens_filtered_negations,
        'lemas_spacy': full_lemmas,
        'stems_snowball': full_stems
    })

# Convert integrated_results to DataFrame for display
df_integrated = pd.DataFrame(integrated_results)
# Display with option to wrap text for better readability
pd.set_option('display.max_colwidth', None)
print('### Resultados Integrados por Documento\n')
display(df_integrated)

# Calculate indicators
num_documents = len(corpus)
total_initial_tokens = len(all_initial_tokens_flat)
total_filtered_tokens = len(all_filtered_tokens_flat)
unique_initial_tokens = len(set(all_initial_tokens_flat))
unique_filtered_tokens = len(set(all_filtered_tokens_flat))

# Ten most frequent lemmas after filtering
lemma_counts = collections.Counter(all_lemmas_filtered_flat)
most_frequent_lemmas = lemma_counts.most_common(10)

print('\n### Indicadores Calculados\n')
print('Criterio para contar tokens y lemas: Se excluyen signos de puntuaci\u00F3n, s\u00EDmbolos, n\u00FAmeros y etiquetas de sustituci\u00F3n (e.g., <URL>, <EMAIL>, <EMOJI>, <MENCION>, <FECHA>, <CANTIDAD>). Esto se aplica tanto a los tokens iniciales como a los filtrados.\n')
print(f"1. N\u00FAmero de documentos: {num_documents}")
print(f"2. Total de tokens (antes del filtrado de stopwords): {total_initial_tokens}")
print(f"3. Total de tokens (despu\u00E9s del filtrado de stopwords, conservando negaciones): {total_filtered_tokens}")
print(f"4. N\u00FAmero de tokens diferentes (antes del filtrado de stopwords): {unique_initial_tokens}")
print(f"5. N\u00FAmero de tokens diferentes (despu\u00E9s del filtrado de stopwords, conservando negaciones): {unique_filtered_tokens}")
print(f"6. Los diez lemas m\u00E1s frecuentes (despu\u00E9s del filtrado de stopwords):")
for lemma, count in most_frequent_lemmas:
    print(f"   - '{lemma}': {count}")

print('\n### Comprobaciones\n')
print('1. Los 16 documentos conservan su identificador y texto original: S\u00ED (ver tabla superior).')
print('2. S\u00ED y si permanecen diferenciados en el texto normalizado: S\u00ED, `s\u00ED` (con tilde) y `si` (sin tilde) se mantienen diferenciados en el procesamiento (`diccionario_normalizacion` es expl\u00EDcito).')
print('3. Las negaciones se conservan en la versi\u00F3n adaptada: S\u00ED (ver columna `tokens_filtrados_negaciones` y definici\u00F3n de `spanish_stopwords_vB`).')
print('4. El valor $350.50 se extrae completo: S\u00ED (ver `informacion_regex` para el documento 6).')
print('5. El contenido de las etiquetas HTML no desaparece: S\u00ED, solo se eliminan las etiquetas, no el texto entre ellas (ver `texto_normalizado_limpio` para el documento 3, que antes ten\u00EDa `<p>`).')
print('6. No se pierden documentos durante el procesamiento: S\u00ED, todos los 16 documentos est\u00E1n presentes en los resultados.')

### Resultados Integrados por Documento



,id,texto_original,informacion_regex,texto_normalizado_limpio,tokens_spacy_full,tokens_spacy_para_conteo,tokens_filtrados_negaciones,lemas_spacy,stems_snowball
0,1,¡¡¡EXCELENTE curso de PLN!!! Aprendí muchísimo 😊😊.,[],¡¡¡excelente curso de pln! aprendí muchísimo <EMOJI> .,"[¡, ¡, ¡, excelente, curso, de, pln, !, aprendí, muchísimo, <, EMOJI, >, .]","[excelente, curso, de, pln, aprendí, muchísimo, <, EMOJI, >]","[excelente, curso, pln, aprendí, muchísimo, <, EMOJI, >]","[¡, ¡, ¡, excelente, curso, de, pln, !, aprender, muchísimo, <, EMOJI, >, .]","[¡, ¡, ¡, excelent, curs, de, pln, !, aprend, muchisim, <, emoji, >, .]"
1,2,No me gustó la explicación de tokenización... fue confusa 😕.,[],no me gustó la explicación de tokenización. fue confusa <EMOJI> .,"[no, me, gustó, la, explicación, de, tokenización, ., fue, confusa, <, EMOJI, >, .]","[no, me, gustó, la, explicación, de, tokenización, fue, confusa, <, EMOJI, >]","[no, gustó, explicación, tokenización, confusa, <, EMOJI, >]","[no, yo, gustar, el, explicación, de, tokenización, ., ser, confuso, <, EMOJI, >, .]","[no, me, gust, la, explic, de, tokeniz, ., fue, confus, <, emoji, >, .]"
2,3,Los profesores explicaron MUY bien.\n ¡Gracias!,[],los profesores explicaron muy bien. ¡gracias!,"[los, profesores, explicaron, muy, bien, ., ¡, gracias, !]","[los, profesores, explicaron, muy, bien, gracias]","[profesores, explicaron, bien, gracias]","[el, profesor, explicar, mucho, bien, ., ¡, gracias, !]","[los, profesor, explic, muy, bien, ., ¡, graci, !]"
3,4,Consulta el material en https://ejemplo.com/pln.,"[{'documento_id': 4, 'tipo': 'URL', 'valor': 'https://ejemplo.com/pln'}]",consulta el material en <url>,"[consulta, el, material, en, <, url, >]","[consulta, el, material, en, <, url, >]","[consulta, material, <, url, >]","[consulta, el, material, en, <, url, >]","[consult, el, material, en, <, url, >]"
4,5,Tengo dudas; escriban a curso@ejemplo.com antes del 25/09/2026.,"[{'documento_id': 5, 'tipo': 'Correo', 'valor': 'curso@ejemplo.com'}, {'documento_id': 5, 'tipo': 'Fecha', 'valor': '25/09/2026'}, {'documento_id': 5, 'tipo': 'Mención', 'valor': '@ejemplo'}]",tengo dudas; escriban a <email> antes del 25/09/2026.,"[tengo, dudas, ;, escriban, a, <, email, >, antes, del, 25/09/2026, .]","[tengo, dudas, escriban, a, <, email, >, antes, del, 25/09/2026]","[dudas, escriban, <, email, >, 25/09/2026]","[tener, duda, ;, escrir, a, <, email, >, antes, del, 25/09/2026, .]","[teng, dud, ;, escrib, a, <, email, >, antes, del, 25/09/2026, .]"
5,6,El taller cuesta $350.50 e incluye 3 sesiones.,"[{'documento_id': 6, 'tipo': 'Cantidad', 'valor': '$350.50'}]",el taller cuesta $350.50 e incluye 3 sesiones.,"[el, taller, cuesta, $, 350.50, e, incluye, 3, sesiones, .]","[el, taller, cuesta, $, e, incluye, sesiones]","[taller, cuesta, $, incluye, sesiones]","[el, taller, costar, $, 350.50, e, incluir, 3, sesión, .]","[el, tall, cuest, $, 350.50, e, inclu, 3, sesion, .]"
6,7,"Los alumnos estaban estudiando, estudiaron y estudiarán Python.",[],"los alumnos estaban estudiando, estudiaron y estudiarán python.","[los, alumnos, estaban, estudiando, ,, estudiaron, y, estudiarán, python, .]","[los, alumnos, estaban, estudiando, estudiaron, y, estudiarán, python]","[alumnos, estudiando, estudiaron, estudiarán, python]","[el, alumno, estar, estudiar, ,, estudiar, y, estudiar, python, .]","[los, alumn, estab, estudi, ,, estudi, y, estudi, python, .]"
7,8,Las niñas llevaron libros y los niños llevaron libretas.,[],las niñas llevaron libros y los niños llevaron libretas.,"[las, niñas, llevaron, libros, y, los, niños, llevaron, libretas, .]","[las, niñas, llevaron, libros, y, los, niños, llevaron, libretas]","[niñas, llevaron, libros, niños, llevaron, libretas]","[el, niña, llevar, libro, y, el, niño, llevar, libreta, .]","[las, niñ, llev, libr, y, los, niñ, llev, libret, .]"
8,9,Ayer fui al laboratorio; el semestre pasado fui representante.,[],ayer fui al laboratorio; el semestre pasado fui representante.,"[ayer, fui, a


### Indicadores Calculados

Criterio para contar tokens y lemas: Se excluyen signos de puntuación, símbolos, números y etiquetas de sustitución (e.g., <URL>, <EMAIL>, <EMOJI>, <MENCION>, <FECHA>, <CANTIDAD>). Esto se aplica tanto a los tokens iniciales como a los filtrados.

1. Número de documentos: 16
2. Total de tokens (antes del filtrado de stopwords): 137
3. Total de tokens (después del filtrado de stopwords, conservando negaciones): 92
4. Número de tokens diferentes (antes del filtrado de stopwords): 99
5. Número de tokens diferentes (después del filtrado de stopwords, conservando negaciones): 74
6. Los diez lemas más frecuentes (después del filtrado de stopwords):
   - '<': 5
   - '>': 5
   - 'EMOJI': 3
   - 'no': 3
   - 'estudiar': 3
   - 'excelente': 2
   - 'curso': 2
   - 'explicar': 2
   - 'duda': 2
   - 'sesión': 2

### Comprobaciones

1. Los 16 documentos conservan su identificador y texto original: Sí (ver tabla superior).
2. Sí y si permanecen diferenciados en el texto n

#Preguntas de Reflexión

1.  **¿Por qué conviene extraer correos y cantidades antes de eliminar caracteres?**
    Conviene extraer correos y cantidades (y otros patrones específicos como URLs, fechas, hashtags, menciones) antes de eliminar caracteres porque el proceso de limpieza y normalización a menudo implica la modificación o eliminación de caracteres especiales, símbolos, puntuación, o la conversión a minúsculas, lo que podría destruir la estructura única de estos elementos. Por ejemplo, si se eliminaran los caracteres `@` o `.` de un correo electrónico, o el `$` de una cantidad monetaria, las expresiones regulares que buscan estos patrones específicos dejarían de funcionar o los identificarían incorrectamente. Al extraerlos primero, se asegura que la información valiosa se capture intacta antes de cualquier transformación que pueda hacerla irreconocible.

2.  **¿Qué información se pierde al quitar acentos?**
    Al quitar los acentos, se pierde información fonética y, en español, también semántica y morfológica. Los acentos son cruciales para:
    *   **Diferenciación de palabras:** Palabras como `sí` (afirmación) y `si` (condicional), `él` (pronombre) y `el` (artículo), `tú` (pronombre) y `tu` (posesivo) tienen significados completamente diferentes que dependen del acento.
    *   **Pronunciación:** Los acentos marcan la sílaba tónica, lo que es fundamental para la correcta pronunciación.
    *   **Tiempos verbales:** `cantó` (pasado) vs. `canto` (presente) o `cantara` (subjuntivo).
    *   **Claridad y ambigüedad:** La ausencia de acentos puede introducir ambigüedad y dificultar la interpretación correcta de un texto, especialmente en tareas como el análisis de sentimientos o la extracción de entidades.

3.  **¿Por qué un emoji puede ser relevante?**
    Un emoji puede ser extremadamente relevante porque transmite una gran cantidad de información emocional y contextual en un formato compacto. En el análisis de opiniones o sentimientos, los emojis son indicadores directos y potentes del estado de ánimo del emisor. Por ejemplo, un 👍 puede indicar aprobación, mientras que un 👎 indica desaprobación. Ignorarlos o eliminarlos sin procesar puede llevar a una interpretación sesgada o incorrecta del sentimiento general de un texto. Transformarlos a etiquetas semánticas (ej. `<EMOJI_POSITIVO>`, `<EMOJI_NEGATIVO>`) permite conservar esta información esencial sin saturar el vocabulario con caracteres especiales.

4.  **¿En qué se diferencian un lema y un stem?**
    La lematización y el stemming son técnicas de normalización morfológica, pero difieren en su objetivo y el resultado:
    *   **Lema (Lematización):** Es la forma base o diccionario de una palabra. Por ejemplo, los lemas de `corriendo`, `corrió`, `correrá` es `correr`. Los lemas son palabras reales y tienen significado lingüístico. Los lematizadores (como spaCy) utilizan diccionarios y análisis morfológico para transformar una palabra a su forma canónica, conservando el contexto semántico.
    *   **Stem (Stemming):** Es una

5.  **¿Por qué conviene lematizar antes de eliminar palabras del contexto?**
    Conviene lematizar antes de eliminar palabras del contexto (especialmente stopwords o para ciertas tareas de reducción de vocabulario) por varias razones:
    *   **Unificación:** La lematización agrupa todas las formas flexionadas de una palabra (ej. 'corriendo', 'corrió', 'correrá' se reducen a 'correr'). Esto consolida el vocabulario y asegura que todas las variantes se traten como la misma entidad semántica antes de tomar decisiones sobre su eliminación. Si eliminamos stopwords antes, podríamos perder la oportunidad de lematizar palabras que, al ser reducidas a su lema, podrían ser consideradas stopwords o ser importantes para el contexto.
    *   **Contexto semántico:** La lematización se basa en el análisis morfológico y el contexto, asegurando que la palabra resultante sea un término válido del diccionario y que su significado se preserve. Esto es crucial si posteriormente se necesita el lema para análisis de sentimiento, topic modeling o extracción de información donde el significado base es relevante.
    *   **Precisión en el filtrado:** Al lematizar primero, se puede aplicar el filtro de stopwords a los lemas en lugar de a las formas superficiales. Esto es más preciso porque una palabra que no es una stopword en su forma original podría serlo una vez lematizada, o viceversa, y evita la eliminación de palabras importantes que tienen formas flexionadas que casualmente coinciden con una stopword.

6.  **¿Qué decisiones cambiarías si el objetivo fuera extraer fechas y precios?**
    Si el objetivo principal fuera extraer fechas y precios, las decisiones de preprocesamiento cambiarían para priorizar la conservación y normalización de estos elementos, en lugar de su eliminación o sustitución genérica:
    *   **Extracción temprana:** La extracción de fechas y precios (usando expresiones regulares como en la Actividad 3) se volvería aún más crítica y se realizaría como uno de los primeros pasos, incluso antes que la eliminación de HTML o la conversión a minúsculas, para asegurar que estos patrones no sean alterados.
    *   **Normalización:** En lugar de simplemente sustituir por etiquetas como `<FECHA>` o `<CANTIDAD>`, se buscaría normalizar estos valores a un formato estándar. Por ejemplo, todas las fechas a `AAAA-MM-DD` y todos los precios a `X.XX` sin símbolo de moneda, o convertir a una moneda común si hay varias.
    *   **No eliminación/sustitución genérica:** No se eliminarían ni se sustituirían por etiquetas genéricas (`_FECHA_`, `_CANTIDAD_`) en el texto final si el objetivo es presentarlos como datos extraídos. En su lugar, se extraerían a un campo estructurado (ej., una columna en una tabla) y el texto original podría mantenerlos, o se sustituirían por una representación estandarizada que conserve el valor.
    *   **Consideración de unidades:** Para precios, se debería considerar la unidad monetaria (dólares, euros) si no está explícita y es relevante. Para fechas, se podría necesitar parsear diferentes formatos (`dd/mm/aaaa`, `mm-dd-aa`, `25 de septiembre de 2026`).
    *   **Menos agresividad en limpieza de puntuación:** Se sería más cuidadoso con la eliminación de puntuación que pueda ser parte integral de un formato de fecha o precio (ej. `/` en `dd/mm/aaaa`, `.` o `,` en `1.500,00`).

7.  **¿La limpieza permite reconocer por sí sola la ironía del documento 16? Explica.**
    **No, la limpieza por sí sola no permite reconocer la ironía en el documento 16.** El documento 16 dice: "Excelente... otra vez no funciona el programa 🙄." La ironía aquí radica en el contraste entre la palabra "Excelente" (que denota algo positivo) y la expresión "otra vez no funciona el programa" junto con el emoji "🙄" (que denota frustración, molestia o incredulidad).

    La limpieza (eliminación de HTML, unificación de espacios, minúsculas, manejo de emojis, etc.) prepara el texto para un análisis, pero no interpreta el significado subyacente ni las figuras retóricas como la ironía:
    *   **"Excelente"**: Después de la limpieza, "excelente" seguirá siendo una palabra positiva. La limpieza no le asignará un significado negativo.
    *   **"otra vez no funciona el programa"**: Esta frase, incluso limpia y con las negaciones conservadas, se interpretará literalmente como una afirmación de un problema.
    *   **Emoji "🙄"**: Aunque la limpieza podría transformar el emoji a una etiqueta como `<EMOJI_FRUSTRACION>` o `<EMOJI_NEGATIVO>`, el sistema necesitaría reglas o modelos de análisis de sentimientos más complejos para conectar esta etiqueta con el significado invertido de "Excelente" en este contexto.

    Para detectar la ironía, se necesitarían técnicas más avanzadas de Procesamiento del Lenguaje Natural (PLN), como:
    *   **Análisis de sentimiento contextual:** Modelos capaces de entender cómo las palabras se relacionan entre sí en una oración para inferir el sentimiento real, incluso cuando hay contradicciones aparentes.
    *   **Detección de incongruencias:** Algoritmos que identifiquen contrastes fuertes entre palabras o frases que suelen indicar polaridades opuestas (ej., una palabra positiva como "excelente" junto a una situación negativa como "no funciona").
    *   **Uso de emojis/emoticones como señales:** Integrar el significado de los emojis como fuertes indicadores de sentimiento que pueden anular el significado literal de las palabras.

    En resumen, la limpieza es un paso fundamental, pero es una etapa preparatoria. La interpretación compleja del lenguaje, como la detección de ironía, requiere modelos lingüísticos y de aprendizaje automático que van más allá de la simple normalización del texto.

#Entregables y evaluación
Entregables
•	Programa .py o cuaderno .ipynb con instrucciones para ejecutarlo y versiones de las bibliotecas utilizadas.

•	Archivo de resultados .json o .csv que conserve el identificador de cada documento.

•	Reporte con tablas comparativas, decisiones de procesamiento, errores detectados y respuestas de reflexión.

•	Demostración de cinco minutos con un comentario nuevo propuesto por la docente. Criterios de evaluación

#Criterio 	Puntos
Identificación de corpus, documento, palabra y token 	10

Detección de ruido y justificación de decisiones 	10

Normalización y eliminación selectiva de caracteres 	15
Expresiones regulares y extracción correcta 	15
Comparación y selección de tokenización 	15
Stopwords y conservación de negaciones 	10
Lematización, stemming y análisis de diferencias 	15
Integración, comprobaciones y explicación de resultados 	10
Total 	100

El criterio central es preparar el texto de acuerdo con el objetivo y justificar las transformaciones, evitando eliminar información útil.

#Referencia curricular
Temario de Procesamiento del Lenguaje Natural, clave TID-2504, Ingeniería en Sistemas Computacionales, Instituto Tecnológico de Ciudad Guzmán, 2025. Unidad 2, subtemas 2.1 a 2.8.
